# Interactive Visual Analytics with Folium

In [1]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [2]:
import folium
import pandas as pd

In [3]:
from folium.plugins import MarkerCluster

from folium.plugins import MousePosition

from folium.features import DivIcon

In [4]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

## 🗺️ Launch Site Locations

In this step, I plotted all SpaceX launch sites on a map using Folium. 

This helps in visualizing the geographical distribution of launch locations.

In [5]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [7]:
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))

marker = folium.map.Marker(
    nasa_coordinate,
    
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

## 🗺️ Launch Sites Visualization

In this step, I added both circles and markers for each launch site using Folium.

The circles highlight the locations, while markers provide interactive information about each launch site.

In [8]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, row in launch_sites_df.iterrows():
    
    coordinate = [row['Lat'], row['Long']]
    label = row['Launch Site']
    
    
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(site_map)
    
    # Add marker
    folium.Marker(
        location=coordinate,
        popup=label,
        icon=folium.Icon(color='red')
    ).add_to(site_map)


site_map.add_child(circle)
site_map.add_child(marker)

In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [10]:
marker_cluster = MarkerCluster()

## 🗺️ Launch Success Visualization

In this step, I visualized launch outcomes using color-coded markers on a map.

Green markers represent successful launches, while red markers represent failed launches. 

Marker clustering was used to handle multiple launches at the same location efficiently.

In [11]:
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})

In [12]:
site_map.add_child(marker_cluster)
for index, row in spacex_df.iterrows():
    
    coordinate = [row['Lat'], row['Long']]
    
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=f"Launch Site: {row['Launch Site']}<br>Success: {row['class']}"
    )
    
    marker_cluster.add_child(marker)

site_map

In [13]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

## 📏 Distance Analysis

In this step, I calculated the distance between the launch site and the nearest coastline using the Haversine formula.

I also visualized this distance on the map using markers and lines.

In [14]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [15]:
launch_site_lat = 28.573255
launch_site_lon = -80.646895

coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon,
    coastline_lat, coastline_lon
)

print("Distance (km):", distance_coastline)

Distance (km): 7.42932194658424


In [16]:
from folium.features import DivIcon

distance_marker = folium.Marker(
    location=[coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>',
    )
)

site_map.add_child(distance_marker)

### PolyLine between a launch site to the selected coastline point



In [17]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(locations=coordinates, weight=2)

site_map.add_child(lines)

site_map

In [18]:
city_lat = 28.5721
city_lon = -80.5853

In [19]:
distance_city = calculate_distance(
    launch_site_lat, launch_site_lon,
    city_lat, city_lon
)

print("Distance to city (km):", distance_city)

Distance to city (km): 6.018172929077765


In [20]:
from folium.features import DivIcon

city_marker = folium.Marker(
    location=[city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:blue;"><b>{distance_city:.2f} KM</b></div>',
    )
)

site_map.add_child(city_marker)

In [21]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [city_lat, city_lon]
]

line = folium.PolyLine(locations=coordinates, weight=2, color='blue')

site_map.add_child(line)

site_map

## 📍 Proximity Analysis

From the map visualization:

- Launch sites are located close to coastlines for efficient rocket launches.
- They are relatively near highways and railways for transportation of materials.
- However, launch sites are located at a safe distance from cities to minimize risk to human populations.

This shows that both accessibility and safety are considered when selecting launch sites.